# Carga dos dados

Os dois CSVs já estão versionados em `data/`, então o notebook roda **offline**.
O bloco do Kaggle é opcional: só é usado se você quiser re-baixar o dataset original.

In [1]:
import os
import shutil

import pandas as pd

pd.set_option("display.width", 200)

## 1. (Opcional) baixar o dataset do Kaggle

Precisa de `~/.kaggle/kaggle.json`. Se você não tem, pule para o passo 2 —
os arquivos em `data/` já são os do dataset.

In [2]:
DATASET = "ilkeryildiz/example-dataset-for-ab-test"
ARQUIVOS = ["control_group.csv", "test_group.csv"]

caminho_destino = os.path.join("..", "data")
os.makedirs(caminho_destino, exist_ok=True)

try:
    import kagglehub

    path = kagglehub.dataset_download(DATASET)
    print("Dataset baixado em:", path)
    for nome in ARQUIVOS:
        shutil.copy(os.path.join(path, nome), os.path.join(caminho_destino, nome))
        print(f"  copiado: {nome}")
except Exception as exc:  # kagglehub ausente, sem credencial ou sem rede
    print("Download ignorado:", type(exc).__name__, exc)
    print("Seguindo com os arquivos que ja estao em data/.")

Download ignorado: ModuleNotFoundError No module named 'kagglehub'
Seguindo com os arquivos que ja estao em data/.


## 2. Carregar os dois grupos

Detalhe que quebrava a análise: **o CSV usa `;` como separador**, não `,`.
Ler com o padrão do pandas colocava a linha inteira numa coluna só, e aí
era preciso um `fix_dataframe()` para consertar depois.

In [3]:
SEP = ";"  # o dataset usa ponto e virgula

df_control = pd.read_csv(os.path.join(caminho_destino, "control_group.csv"), sep=SEP)
df_test = pd.read_csv(os.path.join(caminho_destino, "test_group.csv"), sep=SEP)

for nome, df in (("control", df_control), ("test", df_test)):
    df["Date"] = pd.to_datetime(df["Date"], format="%d.%m.%Y", dayfirst=True)
    df["group"] = nome

df_ab = pd.concat([df_control, df_test], ignore_index=True).sort_values(
    ["group", "Date"]
).reset_index(drop=True)

print("Colunas:", list(df_ab.columns))
print()
df_ab.head()

Colunas: ['Campaign Name', 'Date', 'Spend [USD]', '# of Impressions', 'Reach', '# of Website Clicks', '# of Searches', '# of View Content', '# of Add to Cart', '# of Purchase', 'group']



,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase,group
0,Control Campaign,2019-08-01,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,control
1,Control Campaign,2019-08-02,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,control
2,Control Campaign,2019-08-03,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,control
3,Control Campaign,2019-08-04,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,control
4,Control Campaign,2019-08-05,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,control


In [4]:
df_ab.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Campaign Name        60 non-null     str           
 1   Date                 60 non-null     datetime64[us]
 2   Spend [USD]          60 non-null     int64         
 3   # of Impressions     58 non-null     float64       
 4   Reach                58 non-null     float64       
 5   # of Website Clicks  58 non-null     float64       
 6   # of Searches        58 non-null     float64       
 7   # of View Content    58 non-null     float64       
 8   # of Add to Cart     58 non-null     float64       
 9   # of Purchase        58 non-null     float64       
 10  group                60 non-null     str           
dtypes: datetime64[us](1), float64(7), int64(1), str(2)
memory usage: 6.5 KB


In [5]:
df_ab.groupby("group")[["Spend [USD]", "# of Impressions", "# of Purchase"]].agg(["sum", "mean"]).round(1)

Spend [USD]         # of Impressions           # of Purchase       
                sum    mean              sum      mean           sum   mean
group                                                                      
control       68653  2288.4        3177233.0  109559.8       15161.0  522.8
test          76892  2563.1        2123249.0   73215.5       14869.0  512.7